In [5]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_polizas = 1000

severidades = np.random.gamma(shape=2.0, scale=800, size=n_polizas)

primas = severidades * 0.6 + np.random.normal(loc=0, scale=150, size=n_polizas)
primas = np.clip(primas, 100, None)

print(f"Severidades generadas: {len(severidades)}")
print(f"Primas generadas: {len(primas)}")

Severidades generadas: 1000
Primas generadas: 1000


In [4]:
pd.Series(severidades).describe()

count    1000.000000
mean     1647.944213
std      1122.285679
min        36.735181
25%       820.969063
50%      1383.251199
75%      2193.726145
max      6229.512341
dtype: float64

In [7]:
class RiskLoadingTransformer:
    def __init__(self, recargo_pct=0.15):
        self.recargo_pct = recargo_pct
        self.umbral_ = None

    def fit(self, severidades):
        self.umbral_ = np.percentile(severidades, 90)
       
    def transform(self, primas):
       return np.where(primas > self.umbral_, primas * (1 + self.recargo_pct), primas)

    def fit_transform(self, severidades, primas):
        self.fit(severidades)
        return self.transform(primas)

In [8]:
transformer = RiskLoadingTransformer(recargo_pct=0.20)
resultado = transformer.fit_transform(severidades, primas)
print(resultado[:10])  # ver las primeras 10 primas resultantes

[ 938.01848759  704.87723461  437.78806439  777.51349583 2244.22888081
 1157.38626594  496.53607928 1072.68748641 1007.37710331  304.70673978]


In [9]:
print(f"Umbral aprendido (percentil 90): {transformer.umbral_:.2f}")
print(f"¿Cuántas primas superaron el umbral y recibieron recargo? {(primas > transformer.umbral_).sum()} de {len(primas)}")

Umbral aprendido (percentil 90): 3283.30
¿Cuántas primas superaron el umbral y recibieron recargo? 8 de 1000


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

class RiskLoadingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, recargo_pct=0.15):
        self.recargo_pct = recargo_pct
        self.umbral_ = None

    def fit(self, severidades, y=None):
        self.umbral_ = np.percentile(severidades, 90)
        return self  # clave: fit() debe devolver self

    def transform(self, primas):
        return np.where(primas > self.umbral_, primas * (1 + self.recargo_pct), primas)


In [11]:
transformer = RiskLoadingTransformer(recargo_pct=0.20)
resultado = transformer.fit_transform(severidades, primas)
print(resultado[:10])

[1914.9435119  1195.57178417 1105.8268675  1105.84183547 4463.72583574
 2293.36498477  904.86240139 1975.85157874 1599.16821125  172.73195479]
